MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Obstacle Mask Generation

This notebook loads an elevation sample and markers, then generates an obstacle mask using randomized superellipses. The obstacle mask uses **0** for valid path cells and **1** for forbidden cells.

In [ ]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from passage.utils import load_tensor, TENSOR_SUFFIX
from loguru import logger as loguru_logger

# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from passage.generate import (
    _generate_obstacles_mask,
    _resolve_obstacle_config,
    _resolve_solver_config,
    generate_sample,
    lat_to_str,
    lon_to_str,
)

In [ ]:
# Parameters (papermill)
RESOLUTION = 512
SAMPLE_INDEX = 0
CENTER_LAT = None
CENTER_LON = None
DISABLE_IMAGES = False
OVERRIDE_EXISTING = True
NUM_MASKS_TO_GENERATE = 30

## Configuration

In [ ]:
# Paths
OUTPUTS_DIR = ROOT / "outputs" / "obstacles"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"
CONFIG_PATH = ROOT / "config" / "config.yaml"

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f) or {}

# read obstacles config
obstacles_cfg = config.get("obstacles", {})
OBSTACLES_ENABLED = bool(obstacles_cfg.get("enabled", False))
OBSTACLE_RATIO_MAX = float(obstacles_cfg.get("ratio_max", 0.2))
shape_cfg = obstacles_cfg.get("shape", {})
size_cfg = shape_cfg.get("size", {})
OBSTACLE_SIZE_MIN_RATIO = float(size_cfg.get("min_ratio", 0.20))
OBSTACLE_SIZE_MAX_RATIO = float(size_cfg.get("max_ratio", 0.50))
N_CFG = shape_cfg.get("n", {})
N_DISTRIBUTION_CFG = N_CFG.get("distribution", {})

# generation tuning
GENERATION_CFG = obstacles_cfg.get("generation", {})
obstacle_settings = _resolve_obstacle_config(config)

# Obstacle mask generation settings
NUM_MASKS_MAX = 40  # Upper bound constant (used for display/experiments)
NUM_MASKS_TO_GENERATE = 20  # Number of masks to generate and display (<= NUM_MASKS_MAX)

COST_MODEL = (
    config.get("solver", {}).get("grid", {}).get("cost_model")
    or config.get("solver", {}).get("networkx", {}).get("cost_model")
    or config.get("solver", {}).get("pathfinding", {}).get("cost_model")
    or "elevation"
)

# Solver defaults for sample generation read from config
GEN_SOLVER_CONFIG = _resolve_solver_config(config.get("solver", {}))

In [4]:
# Shape and n distribution config (nested structure)
shape_cfg = obstacles_cfg.get("shape", {})
size_cfg = shape_cfg.get("size", {})
OBSTACLE_SIZE_MIN_RATIO = float(size_cfg.get("min_ratio", 0.20))
OBSTACLE_SIZE_MAX_RATIO = float(size_cfg.get("max_ratio", 0.50))
N_CFG = shape_cfg.get("n", {})
N_DISTRIBUTION_CFG = N_CFG.get("distribution", {})

**Note:** The superellipse exponent `n` may be sampled from different distributions configured under `shape.n.distribution`:

- `log_uniform` (default)
- `uniform`
- `beta`
- `exponential` (new): accepts parameter `lambda` (rate) to bias toward lower values; example: `{"type": "exponential", "lambda": 2.0}`

## Load or Generate Sample

In [ ]:
# Validate required inputs
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

# Load calibration data
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

# Import tensor utilities


def resolve_sample_paths(outputs_dir: Path, resolution: int, index: int, center_lat=None, center_lon=None):
    """Resolve tensor and metadata paths, supporting both compressed (.npy.zst) and legacy (.npy) formats."""
    tensors_root = outputs_dir / str(resolution) / "tensors"
    metadata_root = outputs_dir / str(resolution) / "metadata"
    if center_lat is not None and center_lon is not None:
        lat_dir = lat_to_str(int(center_lat))
        lon_dir = lon_to_str(int(center_lon))
        # Try compressed format first, fall back to legacy
        tensor_path_compressed = tensors_root / lat_dir / lon_dir / f"{index:010d}{TENSOR_SUFFIX}"
        tensor_path_legacy = tensors_root / lat_dir / lon_dir / f"{index:010d}.npy"
        tensor_path = tensor_path_compressed if tensor_path_compressed.exists() else tensor_path_legacy
        metadata_path = metadata_root / lat_dir / lon_dir / f"{index:010d}.yaml"
        return tensor_path, metadata_path
    # Search for tensor with either suffix
    tensor_matches_compressed = list(tensors_root.rglob(f"{index:010d}{TENSOR_SUFFIX}"))
    tensor_matches_legacy = list(tensors_root.rglob(f"{index:010d}.npy"))
    tensor_matches = tensor_matches_compressed or tensor_matches_legacy
    metadata_matches = list(metadata_root.rglob(f"{index:010d}.yaml"))
    tensor_path = tensor_matches[0] if tensor_matches else tensors_root / f"{index:010d}{TENSOR_SUFFIX}"
    metadata_path = metadata_matches[0] if metadata_matches else metadata_root / f"{index:010d}.yaml"
    return tensor_path, metadata_path


def load_tensor_compat(path: Path, resolution: int) -> np.ndarray:
    """Load a tensor from either compressed (.npy.zst) or legacy (.npy) format."""
    if path.suffix == ".zst" or str(path).endswith(TENSOR_SUFFIX):
        return load_tensor(path, (resolution, resolution, 3), "float32")
    else:
        return np.load(path)


tensor_path, metadata_path = resolve_sample_paths(OUTPUTS_DIR, RESOLUTION, SAMPLE_INDEX, CENTER_LAT, CENTER_LON)

if OVERRIDE_EXISTING or not tensor_path.exists() or not metadata_path.exists():
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    args = (
        SAMPLE_INDEX,
        TILES_DIR,
        OUTPUTS_DIR,
        RESOLUTION,
        tile_stats_df,
        global_min,
        global_max,
        DISABLE_IMAGES,
        GEN_SOLVER_CONFIG,
        obstacle_settings,
    )
    index, success, message, duration = generate_sample(args)
    print(f"Generate sample: success={success}, message='{message}', duration={duration:.2f}s")
    if not success:
        raise RuntimeError("Sample generation failed. Try a different resolution or re-run calibration.")
    # Re-resolve paths after generation (new format)
    tensor_path, metadata_path = resolve_sample_paths(OUTPUTS_DIR, RESOLUTION, SAMPLE_INDEX, CENTER_LAT, CENTER_LON)
else:
    print(f"Using existing tensor: {tensor_path}")

tensor = load_tensor_compat(tensor_path, RESOLUTION)
elevation_norm = tensor[:, :, 0]
marker_channel = tensor[:, :, 1]
obstacle_channel = tensor[:, :, 2] if tensor.shape[2] > 2 else np.zeros_like(marker_channel)
elevation_raw = elevation_norm * (global_max - global_min) + global_min

print(f"Loaded tensor shape: {tensor.shape}")

2026-01-19 22:07:01.097 | INFO     | passage.generate:generate_sample:1243 - Sample 0: success=True, elevation_attempts=1, marker_retries=1, path_length_no_obstacles=139, path_length_with_obstacles=139, grid_shape=(10800,10800), duration=3.38s


Generate sample: success=True, message='Generated sample 0', duration=3.38s
Loaded tensor shape: (256, 256, 3)


## Obstacle Generation

In [6]:
# Using _generate_obstacles_mask from generate.py
# The function signature is:
# _generate_obstacles_mask(
#     shape, ratio_max, size_min_ratio, size_max_ratio, obstacles_cfg,
#     marker_positions=None, rng=None, max_attempts=None, generation_cfg=None
# ) -> tuple[np.ndarray, float]
#
# Note: obstacles_cfg is required and contains the full obstacles config dict

In [7]:
start_positions = np.argwhere(np.isclose(marker_channel, 0.0))
end_positions = np.argwhere(marker_channel > 0.5)
marker_positions = np.vstack([start_positions, end_positions])

# Respect the notebook-level constants and ensure we don't exceed the defined maximum
NUM_MASKS = min(NUM_MASKS_TO_GENERATE, NUM_MASKS_MAX)

masks = []
ratios = []
for i in range(NUM_MASKS):
    # Use _generate_obstacles_mask from generate.py
    mask, ratio = _generate_obstacles_mask(
        shape=elevation_raw.shape,
        ratio_max=OBSTACLE_RATIO_MAX,
        size_min_ratio=OBSTACLE_SIZE_MIN_RATIO,
        size_max_ratio=OBSTACLE_SIZE_MAX_RATIO,
        obstacles_cfg=obstacles_cfg,
        marker_positions=marker_positions,
        generation_cfg=GENERATION_CFG,
    )
    masks.append(mask)
    ratios.append(ratio)

masks_arr = np.stack(masks, axis=0)

# Choose the first generated mask as the active obstacles_mask for downstream cells
obstacles_mask = masks_arr[0]
tensor_with_obstacles = np.stack([elevation_norm, marker_channel, obstacles_mask], axis=-1)

print(f"Generated {NUM_MASKS} obstacle masks (displaying {NUM_MASKS}).")
print(f"Target obstacle ratio stats: min={min(ratios):.3f}, mean={np.mean(ratios):.3f}, max={max(ratios):.3f}")
print(f"Tensor with obstacles shape: {tensor_with_obstacles.shape}")

# Display the masks in a compact grid gallery
n = masks_arr.shape[0]
cols = min(5, n)
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
if cols == 1:
    axes = axes.reshape(-1, 1)

for idx in range(n):
    r = idx // cols
    c = idx % cols
    axes[r, c].imshow(masks_arr[idx], cmap="gray", origin="upper")
    axes[r, c].set_title(f"Mask {idx} (r={ratios[idx]:.3f})", fontsize=9)
    axes[r, c].axis("off")

# Hide empty subplots
for idx in range(n, rows * cols):
    r = idx // cols
    c = idx % cols
    axes[r, c].axis("off")

plt.tight_layout()
plt.show()

Generated 20 obstacle masks (displaying 20).
Target obstacle ratio stats: min=0.014, mean=0.145, max=0.291
Tensor with obstacles shape: (256, 256, 3)


## Pathfinding (grid:astar)

In [8]:
start_positions = np.argwhere(np.isclose(marker_channel, 0.0))
end_positions = np.argwhere(marker_channel > 0.5)
if start_positions.shape[0] < 1 or end_positions.shape[0] < 1:
    raise RuntimeError("Expected start and end markers in marker channel.")

start = tuple(start_positions[0])
end = tuple(end_positions[0])

# Resolve solver config from config.yaml (supports common + backend sections)
solver_config = _resolve_solver_config(config.get("solver", {}))
# Ensure sensible defaults
solver_config.setdefault("diagonal_movement", "always")
solver_config.setdefault("weight", 30.0)

100.0